# MMELON Single-View Fine-Tuning

Fine-tune one encoder of the pretrained MMELON model (text, image or graph) on its own for
**binary classification** of molecules (e.g. hit / non-hit) from SMILES.

**Environment:** run with the `biomed-multiview` kernel (see [installation-troubleshooting.md](installation-troubleshooting.md)).

**Inputs:** a train CSV and a test CSV, each with a SMILES column and a 0/1 label column.

**Single view:** trains one MMELON encoder on its own, with no fusion: `VIEW = "text"` (SMILES transformer,
MoLFormer architecture), `"image"` (ResNet18, ImageMol) or `"graph"` (graph transformer, TokenGT). The encoder
starts from its weights in the pretrained MMELON checkpoint (`ibm-research/biomed.sm.mv-te-84m`) and is
followed by the same MLP prediction head as MMELON. Everything else (inputs, metrics, model selection, outputs)
is the same as the multi-view notebooks, so results are directly comparable.
For the multi-view model (MMELON), use the matching `MMELON_*` notebook without `singleview` in its name.

**Metrics**
- **hit@K** (hit@200, hit@100, hit@50): rank the test molecules by predicted score; hit@K is how many
  of the top K have label 1. If the test set has fewer than K molecules, all of them are counted.
- Standard classification metrics: AUROC, precision, recall, specificity and loss.

**How the model is selected:** train on the train file for `EPOCHS` epochs, scoring the test file after every
epoch. Epochs are ranked by `SELECT_METRICS` (default: hit@200, then hit@100, then hit@50, then AUROC to
break ties), and the best epoch is kept as the fine-tuned model.

Because the test file is used to pick the epoch, its score is slightly optimistic.
For an unbiased estimate, evaluate the final model on a separate held-out file.

For k-fold cross-validation, use [MMELON_singleview_finetuning_cv.ipynb](MMELON_singleview_finetuning_cv.ipynb) instead.

**Output** (in `biomed-multi-view/output/`, which is gitignored):
```
output/
├── base_models/biomed.sm.mv-te-84m/     # pretrained model from Hugging Face (downloaded once)
└── finetuned/<RUN_NAME>_<VIEW>_<timestamp>/
    ├── config.json                      # settings used for this run
    ├── data/                            # data_train.csv, data_val.csv (= the test file)
    ├── logs/metrics.csv                 # metrics per epoch
    ├── metrics_per_epoch.csv            # test metrics per epoch
    ├── selection.json                   # selected epoch and its test metrics
    └── best_model.ckpt                  # the fine-tuned model
```

## Steps
1. Inputs and settings
2. Prepare data
3. Download the base model
4. Data loaders
5. Build the model
6. Train and select the best epoch

## 1. Inputs and settings

**Edit these before running.**

In [ ]:
from pathlib import Path

# Name of this run; outputs go to output/finetuned/<RUN_NAME>_<VIEW>_<timestamp>/
RUN_NAME = "temp-sample"

# Encoder to train on its own: "text" (SMILES transformer, MoLFormer), "image" (ResNet18, ImageMol)
# or "graph" (graph transformer, TokenGT)
VIEW = "text"

# ── Data ──────────────────────────────────────────────────────────────────────
DATA_DIR = Path.home() / "Datasets" / "temp-sample"

TRAIN_PATH = DATA_DIR / "data_train.csv"
TEST_PATH = DATA_DIR / "data_test.csv"

SMILES_COL = "smiles"
LABEL_COL = "label"

In [ ]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model ID

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 10
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

# hit@K = number of label-1 molecules among the K highest-scored test molecules
HIT_KS = [200, 100, 50]

# Model selection, in priority order: the first metric decides, later ones break ties.
# Choose from hit@K (for K in HIT_KS), "rocauc", "precision", "recall", "specificity", "loss".
SELECT_METRICS = ["hit@200", "hit@100", "hit@50", "rocauc"]

MAX_PER_CLASS = None      # e.g. 100 for a quick test run; None uses all data
NUM_WORKERS = min(4, os.cpu_count())

In [ ]:
import json
import shutil
from datetime import datetime

import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "output"
BASE_MODEL_DIR = OUTPUT_DIR / "base_models" / BASE_MODEL.split("/")[-1]
RUN_DIR = OUTPUT_DIR / "finetuned" / f"{RUN_NAME}_{VIEW}_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DATA_DIR = RUN_DIR / "data"

assert VIEW in ("text", "image", "graph"), f"VIEW must be text, image or graph (for multiview use MMELON_finetuning.ipynb), got {VIEW!r}"
for path in (TRAIN_PATH, TEST_PATH):
    assert Path(path).exists(), f"File not found: {path}"

# One checkpoint per epoch (~0.35 GB) is kept until the best epoch is chosen
needed_gb = 0.35 * EPOCHS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > needed_gb + 2, f"Need ~{needed_gb:.0f} GB free for checkpoints, only {free_gb:.0f} GB available"

RUN_DATA_DIR.mkdir(parents=True, exist_ok=True)

config = dict(
    train_path=str(TRAIN_PATH), test_path=str(TEST_PATH),
    smiles_col=SMILES_COL, label_col=LABEL_COL, base_model=BASE_MODEL,
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, seed=SEED, hit_ks=HIT_KS, select_metrics=SELECT_METRICS,
    max_per_class=MAX_PER_CLASS, view=VIEW,
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Repo root  : {REPO_ROOT}")
print(f"Base model : {BASE_MODEL_DIR}")
print(f"Run folder : {RUN_DIR}")
print(f"View       : {VIEW}")
print(f"Disk       : ~{needed_gb:.1f} GB needed during training, {free_gb:.0f} GB free")
print(f"Device     : {'GPU' if torch.cuda.is_available() else 'CPU (training will be slow)'}")

## 2. Prepare data

The MMELON data pipeline reads `data_train.csv` and `data_val.csv` from one folder, with columns
`smiles` and `label`, and scores `data_val.csv` after every epoch. This step copies the train file to
`data_train.csv` and the **test file to `data_val.csv`** in the run's `data/` folder.

In [ ]:
import pandas as pd


def load_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    missing = {SMILES_COL, LABEL_COL} - set(df.columns)
    assert not missing, f"{path} is missing columns {missing}; found {df.columns.tolist()}"
    df = df[[SMILES_COL, LABEL_COL]].rename(columns={SMILES_COL: "smiles", LABEL_COL: "label"})
    n_before = len(df)
    df = df.dropna()
    df = df[df["smiles"].astype(str).str.strip() != ""]
    if len(df) < n_before:
        print(f"{path.name}: dropped {n_before - len(df)} rows with empty SMILES or label")
    df["label"] = df["label"].astype(int)
    assert set(df["label"].unique()) <= {0, 1}, f"{path} labels must be 0/1"
    return df.reset_index(drop=True)


def limit_per_class(df: pd.DataFrame, n: int) -> pd.DataFrame:
    parts = [g.sample(n=min(n, len(g)), random_state=SEED) for _, g in df.groupby("label")]
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)


summary = []
for name, path, out_name in [("train", TRAIN_PATH, "data_train.csv"), ("test", TEST_PATH, "data_val.csv")]:
    df = load_split(path)
    if MAX_PER_CLASS is not None:
        df = limit_per_class(df, MAX_PER_CLASS)
    df.to_csv(RUN_DATA_DIR / out_name, index=False)
    summary.append(dict(split=name, samples=len(df), positives=int(df["label"].sum()),
                        positive_pct=round(100 * df["label"].mean(), 2)))

pd.DataFrame(summary)

## 3. Download the base model

Downloaded once into `output/base_models/`; later runs reuse it.

In [ ]:
from huggingface_hub import snapshot_download

if (BASE_MODEL_DIR / "model.safetensors").exists():
    print(f"Using existing base model: {BASE_MODEL_DIR}")
else:
    print(f"Downloading {BASE_MODEL} ...")
    snapshot_download(repo_id=BASE_MODEL, local_dir=BASE_MODEL_DIR)
    print(f"Saved to: {BASE_MODEL_DIR}")

## 4. Data loaders

Each SMILES is converted into the input the chosen view needs: text tokens, a 2D image or a molecular graph.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel
from bmfm_sm.core.data_modules.namespace import TaskType
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline

# Per view: model class and settings, data pipeline, input modality, and the prefix of that
# view's encoder weights inside the MMELON checkpoint
VIEW_SPECS = {
    "text": dict(
        model_class="bmfm_sm.predictive.modules.text_models.TextModel",
        model_params={},
        pipeline=TextFinetuneDataPipeline,
        modalities=["TEXT_MODEL"],
        weight_prefix="model_text.",
    ),
    "image": dict(
        model_class="bmfm_sm.predictive.modules.image_models.ImageModel",
        model_params={"baseModel": "ResNet18"},
        pipeline=ImageFinetuneDataPipeline,
        modalities=["IMAGE_MODEL"],
        weight_prefix="model_image.",
    ),
    "graph": dict(
        model_class="bmfm_sm.predictive.modules.graph_2d_models.Graph2dModel",
        model_params={"encoder_embed_dim": 512},
        pipeline=Graph2dFinetuneDataPipeline,
        modalities=["GRAPH_2D_MODEL"],
        weight_prefix="model_graph.",
    ),
}


def dataset_args_for(view: str) -> dict:
    return {
        "task_type": TaskType.CLASSIFICATION,
        "num_tasks": 1,
        "modalities": VIEW_SPECS[view]["modalities"],
        "smiles_col": "smiles",
        "label_cols": ["label"],
        "split_col": None,
    }


def view_of_checkpoint(path) -> str:
    # Which view a fine-tuned checkpoint was trained on, from the model class saved in it
    model_class = torch.load(path, map_location="cpu")["hyper_parameters"]["base_model_class"]
    matches = [v for v, s in VIEW_SPECS.items() if s["model_class"] == model_class]
    assert matches, f"{path} is not a single-view model ({model_class}); use the MMELON_* notebooks for multiview"
    return matches[0]


def to_device(batch):
    # Batches are dicts (multiview, graph) or tuples (text, image)
    if isinstance(batch, dict):
        return {k: to_device(v) for k, v in batch.items()}
    if isinstance(batch, (list, tuple)):
        return type(batch)(to_device(v) for v in batch)
    return batch.to(DEVICE) if torch.is_tensor(batch) else batch

pl.seed_everything(SEED)
SPEC = VIEW_SPECS[VIEW]


def make_loader(stage: str, shuffle: bool) -> DataLoader:
    dataset = SPEC["pipeline"](
        data_dir=str(RUN_DATA_DIR), dataset_args=dataset_args_for(VIEW), stage=stage
    )
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)


train_loader = make_loader("train", shuffle=True)
test_loader = make_loader("val", shuffle=False)  # "val" stage reads data_val.csv = the test file

for name, loader in [("train", train_loader), ("test", test_loader)]:
    print(f"{name:5s}: {len(loader.dataset):6,} samples, {len(loader):5,} batches")

## 5. Build the model

Creates the fine-tuning model (the chosen encoder followed by an MLP prediction head) and loads that
encoder's pretrained weights from the MMELON checkpoint. The prediction head starts from random weights.
The model is extended to also log hit@K after every epoch.

In [ ]:
class HitAtKFineTuneModule(FineTuneLightningModule):
    # Adds hit@K to the scores logged after every epoch (as val_hit@K)

    def on_validation_epoch_start(self):
        super().on_validation_epoch_start()
        self._val_scores, self._val_labels = [], []

    def validation_step(self, batch, batch_idx):
        loss, y, y_hat = self._shared_step(batch, "val")
        self._val_scores.append(y_hat.detach().float().reshape(-1).cpu())
        self._val_labels.append(y.detach().reshape(-1).cpu())
        return loss

    def on_validation_epoch_end(self):
        super().on_validation_epoch_end()
        scores, labels = torch.cat(self._val_scores), torch.cat(self._val_labels)
        top = torch.argsort(scores, descending=True)
        for k in HIT_KS:
            self.log(f"val_hit@{k}", float((labels[top[:k]] == 1).sum()))


lightning_module = HitAtKFineTuneModule(
    base_model_class=SPEC["model_class"],
    model_params=SPEC["model_params"],
    task_type="classification",
    num_tasks=1,
    checkpoint_path=None,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    finetuning_args={
        "weight_freeze": "unfrozen",
        "initialization": "default",
        "head_arch": "mlp",
        "use_norm": True,
        "head_dropout": 0.2,
    },
)

pretrained_state = SmallMoleculeMultiViewModel.from_pretrained(
    LateFusionStrategy.ATTENTIONAL, model_path=str(BASE_MODEL_DIR), huggingface=True
).state_dict()
# Load only this view's encoder from the MMELON checkpoint (weight names without the view prefix)
prefix = SPEC["weight_prefix"]
encoder_state = {k[len(prefix):]: v for k, v in pretrained_state.items() if k.startswith(prefix)}
result = lightning_module.model.load_state_dict(encoder_state, strict=False)
del pretrained_state, encoder_state

# strict=False would silently skip mismatched weights, so check explicitly
assert not result.missing_keys and not result.unexpected_keys, (
    f"Pretrained weights did not match: missing={result.missing_keys[:5]}, "
    f"unexpected={result.unexpected_keys[:5]}"
)
print(f"Pretrained {VIEW} weights loaded (all keys matched).")

## 6. Train and select the best epoch

After every epoch the model is scored on the test file and a weights-only checkpoint is saved.
Epochs are then ranked by `SELECT_METRICS` in order (the first metric decides, later ones break ties);
the best epoch is kept as `best_model.ckpt` and the other checkpoints are deleted.
Metrics prefixed `val_` in the logs are scores on the **test** file.

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

checkpoint_callback = ModelCheckpoint(
    dirpath=RUN_DIR / "checkpoints",
    filename="epoch{epoch:02d}",
    auto_insert_metric_name=False,
    save_top_k=-1,              # keep every epoch until selection
    every_n_epochs=1,
    save_weights_only=True,
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    callbacks=[checkpoint_callback],
    logger=CSVLogger(save_dir=RUN_DIR, name="logs", version=""),
    default_root_dir=RUN_DIR,
    accelerator="auto",
    devices=1,
    log_every_n_steps=10,
    num_sanity_val_steps=0,
    enable_model_summary=True,
)

In [ ]:
%%time
trainer.fit(lightning_module, train_loader, test_loader)

In [ ]:
# Test metrics per epoch
logged = pd.read_csv(RUN_DIR / "logs" / "metrics.csv")
metrics = logged.dropna(subset=["val_loss"])
metric_cols = [c for c in metrics.columns if c.startswith("val_")]
metrics = metrics[["epoch"] + metric_cols].rename(columns=lambda c: c.removeprefix("val_"))
metrics = metrics.groupby("epoch", as_index=False).first()  # one row per epoch
metrics["epoch"] = metrics["epoch"].astype(int)

# hit@K first (in HIT_KS order), then the standard metrics
hits = [f"hit@{k}" for k in HIT_KS]
metric_names = hits + [c for c in metrics.columns if c not in hits and c != "epoch"]
metrics = metrics[["epoch"] + metric_names]
metrics.to_csv(RUN_DIR / "metrics_per_epoch.csv", index=False)

missing = [m for m in SELECT_METRICS if m not in metric_names]
assert not missing, f"SELECT_METRICS {missing} not logged; available: {metric_names}"

# Sort by each selection metric in priority order (loss: lower is better); ties keep the earlier epoch
ranked = metrics.sort_values(SELECT_METRICS, ascending=[m == "loss" for m in SELECT_METRICS], kind="mergesort")
best_metrics = ranked.iloc[0]
best_epoch = int(best_metrics["epoch"])

# Keep the best epoch's checkpoint as best_model.ckpt and delete the others
best_path = RUN_DIR / "best_model.ckpt"
shutil.move(RUN_DIR / "checkpoints" / f"epoch{best_epoch:02d}.ckpt", best_path)
shutil.rmtree(RUN_DIR / "checkpoints")

selection = dict(
    select_metrics=SELECT_METRICS,
    best_epoch=best_epoch,
    epochs_trained=EPOCHS,
    test_metrics={m: float(best_metrics[m]) for m in metric_names},
    model=str(best_path),
)
(RUN_DIR / "selection.json").write_text(json.dumps(selection, indent=2))

print(f"Selected epoch {best_epoch}:")
for m in SELECT_METRICS:
    print(f"  {m:12s} {best_metrics[m]:.4f}")
print(f"Fine-tuned model: {best_path}\n")
display(metrics.round(4))

In [ ]:
# Files produced by this run
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(RUN_DIR)):60s} {path.stat().st_size / 1e6:8.2f} MB")